# DESDEO demo: investing in a summer cabin's electricity system

A cabin owner considers buying a **battery** and **solar panels**. Investing more lowers the
electricity bill (less buying from the grid, more selling to it), so the two objectives conflict:

- $f_1$: electricity cost over the summer (€), negative when selling earns more than buying costs
- $f_2$: investment cost (€)

We model the problem in DESDEO, look at the trade-off, take two NIMBUS steps, and finally save the
problem to the database so the same problem can be solved in the web GUI.

In [ ]:
%%capture
# Gurobi prints license details the first time it starts; keep them off the screen
import contextlib

with contextlib.suppress(ImportError):
    import gurobipy

    gurobipy.Model()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from desdeo.mcdm import nimbus_generate_starting_point, nimbus_solve_sub_problems
from desdeo.problem.testproblems import summer_cabin_battery_problem
from desdeo.tools import add_epsilon_constraints, guess_best_solver, payoff_table_method

## 1. The problem

In [ ]:
problem = summer_cabin_battery_problem()

pd.DataFrame(
    [
        {
            "symbol": v.symbol,
            "name": v.name,
            "type": v.variable_type.value,
            "shape": getattr(v, "shape", None) or "scalar",
        }
        for v in problem.variables
    ]
)

Three **design** decisions (battery yes/no, its capacity, number of panels) and five hourly
**operation** variables over the whole summer. The objectives are stored as MathJSON:

In [ ]:
from pprint import pprint

for obj in problem.objectives:
    print(f"{obj.symbol}: {obj.name} [{obj.unit}], minimize={not obj.maximize}")
    pprint(obj.func)



In [ ]:
solver = guess_best_solver(problem)
solver

DESDEO picks a solver that fits the problem: this is a mixed-integer linear problem, so a MILP
solver is chosen.

## 2. Ideal and nadir

The ideal and nadir points give the ranges of the objectives on the Pareto front. NIMBUS needs them
to scale the objectives, and the web GUI only lets us open a problem in NIMBUS once they are known.
Our problem does not have them yet:

In [ ]:
{obj.symbol: {"ideal": obj.ideal, "nadir": obj.nadir} for obj in problem.objectives}

We estimate them with the payoff table method. A `Problem` is immutable, so
`update_ideal_and_nadir` returns a new problem with the values filled in.

In [ ]:
ideal, nadir = payoff_table_method(problem)
problem = problem.update_ideal_and_nadir(ideal, nadir)

pd.DataFrame({"ideal": ideal, "nadir": nadir})

Without any investment the bill is about 283 €. With the maximum investment of about 25 000 €
the cabin would earn about 170 € over the summer. Neither extreme is attractive on its own.

## 3. The trade-off

With only two objectives we can afford to compute the whole Pareto front with the
epsilon-constraint method: minimize the bill while limiting the investment. With more objectives
this quickly becomes too expensive and too hard to look at, which is why we use interactive methods.

In [ ]:
front = []
for budget in range(0, 25_001, 1_000):
    eps_problem, target, _ = add_epsilon_constraints(
        problem, "target", {"f_2": "f_2_eps"}, objective_symbol="f_1", epsilons={"f_2": budget}
    )
    result = guess_best_solver(eps_problem)(eps_problem).solve(target)
    front.append({**result.optimal_objectives, "battery": round(result.optimal_variables["y"]) == 1})

front = pd.DataFrame(front)


def plot_front(ax):
    ax.plot(front["f_2"], front["f_1"], "-", color="lightgray", zorder=0)
    for has_battery, group in front.groupby("battery"):
        label = "with battery" if has_battery else "panels only"
        ax.plot(group["f_2"], group["f_1"], "o", color="gray" if has_battery else "silver", label=label, zorder=1)
    ax.set_xlabel("Investment cost $f_2$ (€)")
    ax.set_ylabel("Electricity cost $f_1$ (€)")


fig, ax = plt.subplots(figsize=(7, 4))
plot_front(ax)
ax.legend();

The front already tells a story: below about 12 000 € a battery does not pay off, and the cheapest
designs only have solar panels.

## 4. NIMBUS

NIMBUS starts from one Pareto optimal solution. The decision maker then **classifies** the
objectives: which should improve, which may get worse, and how much.

In [ ]:
def summary(result):
    """Objective values and design decisions of a solution."""
    x = result.optimal_variables
    return {
        "electricity cost f_1 (€)": round(result.optimal_objectives["f_1"], 1),
        "investment f_2 (€)": round(result.optimal_objectives["f_2"]),
        "battery (kWh)": round(x["E"], 1) if round(x["y"]) else 0,
        "solar panels": round(x["n"]),
    }


start = nimbus_generate_starting_point(problem)
pd.DataFrame([summary(start)], index=["start"])

### Step 1

*"I don't want to pay for electricity at all. I can invest up to 15 000 € for that."*

- $f_1$: improve to 0 €
- $f_2$: may worsen until 15 000 €

In [ ]:
step1 = nimbus_solve_sub_problems(
    problem,
    current_objectives=start.optimal_objectives,
    reference_point={"f_1": 0, "f_2": 15_000},
    num_desired=3,
)
pd.DataFrame([summary(r) for r in step1], index=["NIMBUS", "STOM", "ASF"])

All three solutions buy a battery. The decision maker likes the STOM solution best: it earns a
little and is the cheapest of the three.

### Step 2

*"Could I spend less, about 12 500 €, and still not pay for electricity?"*

- $f_2$: improve to 12 500 €
- $f_1$: may worsen until 0 €

In [ ]:
chosen = step1[1]

step2 = nimbus_solve_sub_problems(
    problem,
    current_objectives=chosen.optimal_objectives,
    reference_point={"f_1": 0, "f_2": 12_500},
    num_desired=3,
)
pd.DataFrame([summary(r) for r in step2], index=["NIMBUS", "STOM", "ASF"])

The decision maker is happy with the STOM solution: a 14 kWh battery and 30 panels for about
12 300 €, and a small profit from selling electricity.

### Where did we go?

In [ ]:
from itertools import pairwise

final = step2[1]

fig, ax = plt.subplots(figsize=(7, 4))
plot_front(ax)
path = [("start", start), ("step 1", chosen), ("step 2", final)]
for (label, r), color in zip(path, ["tab:blue", "tab:orange", "tab:green"], strict=True):
    ax.plot(r.optimal_objectives["f_2"], r.optimal_objectives["f_1"], "o", ms=10, color=color, label=label)
for (_, a), (_, b) in pairwise(path):
    ax.annotate(
        "",
        xy=(b.optimal_objectives["f_2"], b.optimal_objectives["f_1"]),
        xytext=(a.optimal_objectives["f_2"], a.optimal_objectives["f_1"]),
        arrowprops={"arrowstyle": "->"},
    )
ax.set_xlim(8_000, 16_000)
ax.set_ylim(-75, 60)
ax.legend();

The chosen design also comes with an hourly operating plan. The first week of it:

In [ ]:
week = slice(0, 7 * 24)
x = final.optimal_variables

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 4.5), sharex=True)
ax1.plot(x["buy"][week], label="bought")
ax1.plot(x["sell"][week], label="sold")
ax1.set_ylabel("kWh/h")
ax1.legend()
ax2.plot(x["soc"][week], color="tab:green")
ax2.set_ylabel("battery (kWh)")
ax2.set_xlabel("hour");

## 5. Share the problem as JSON

A `Problem` is plain, serializable data. Saved as JSON, together with its ideal and nadir, it can be
sent to a colleague, kept under version control, or uploaded in the web GUI under
**Problems → Define → Upload JSON**.

In [ ]:
from pathlib import Path

from desdeo.problem import Problem

demo_problem = problem.model_copy(
    update={
        "name": "Summer cabin electricity (demo 2026)",
        "description": "Battery and solar panel investment for a summer cabin: electricity cost vs. investment cost.",
    }
)

json_path = Path("summer_cabin_demo_2026.json")
demo_problem.save_to_json(json_path)

print(f"{json_path}: {json_path.stat().st_size / 1e6:.1f} MB")
print("Loads back unchanged:", Problem.load_json(json_path) == demo_problem)

## 6. Save the problem for the web GUI

We can also write the problem straight into the web API's database, and attach a short solution
description that the GUI shows next to each solution.

Running this cell again does not add a duplicate.

In [ ]:
from sqlmodel import Session, create_engine, select

from desdeo.api.models import ProblemDB, User
from desdeo.api.models.problem import DescriptionPart, ProblemMetaDataDB, SolutionDescriptionMetaData

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
engine = create_engine(f"sqlite:///{repo_root / 'desdeo' / 'api' / 'test.db'}")

description = [
    DescriptionPart(label="Battery installed", symbol="y", format_spec=".0f"),
    DescriptionPart(label="Battery capacity", symbol="E", format_spec=".1f", suffix=" kWh"),
    DescriptionPart(label="Solar panels", symbol="n", format_spec=".0f"),
    DescriptionPart(label="Investment cost", symbol="f_2", format_spec=".0f", suffix=" €"),
    DescriptionPart(label="Electricity cost", symbol="f_1", format_spec=".2f", suffix=" €"),
]

with Session(engine) as session:
    analyst = session.exec(select(User).where(User.username == "analyst")).one()
    problem_db = session.exec(select(ProblemDB).where(ProblemDB.name == demo_problem.name)).first()
    if problem_db is None:
        problem_db = ProblemDB.from_problem(demo_problem, analyst)
        session.add(problem_db)
        session.commit()
        session.refresh(problem_db)

        metadata = ProblemMetaDataDB(problem_id=problem_db.id, problem=problem_db)
        session.add(metadata)
        session.commit()
        session.refresh(metadata)
        session.add(
            SolutionDescriptionMetaData(
                metadata_id=metadata.id, parts=[p.model_dump() for p in description], separator="\n"
            )
        )
        session.commit()
        print(f"Saved '{problem_db.name}' with id {problem_db.id}.")
    else:
        print(f"'{problem_db.name}' is already in the database with id {problem_db.id}.")

Next: log in to the web GUI as `analyst`, pick **Summer cabin electricity (demo 2026)** and
solve it with NIMBUS there.